# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/20Aarya05/FlyrankAI_1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### **Data Contract (5 Plain-words Answers):**

1. **What one row means for your lane:**
   One row represents a single pseudonymized content item (page) for a specific client site, aggregated over a specific feature time window (e.g. 15 or 30 days).

2. **Which table(s) you'll use:**
   - `dim_clients` (client status, history boundaries)
   - `dim_content` (page metadata: word counts, page types, backlinks)
   - `fact_content_daily_performance` (partitioned by month; daily impressions, clicks, avg position, and GA4 engagement sessions)
   - `fact_content_query_90d` (query-level diversity, concentration, tail share)

3. **Which time window:**
   We split our timeline at a decision point (e.g. `2026-03-15`). Features are aggregated over the trailing baseline window (e.g. `2026-03-01` to `2026-03-15`), and labels are computed over the future outcome window (e.g. `2026-03-16` to `2026-03-31`).

4. **What you'd predict or rank (label or proxy):**
   We predict/rank content pages based on the proxy label `is_declining`. This label evaluates to `1` (declining) if future search impressions in the outcome window drop by more than 20% compared to the baseline feature window, and `0` otherwise.

5. **One thing you deliberately exclude:**
   We deliberately exclude the future-outcome impressions (`imp_future15` / `imp_last30`) and clicks (`clk_future15`) from our feature set, as well as `trend_direction` and `trend_pct` since they directly summarize the future label period and cause target leakage. We also exclude client and content ID hashes from model features (keeping them only as context to avoid overfitting to specific IDs).

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os

REL = 'data/raw/warehouse_local'
print("Paths and packages loaded successfully.")


Paths and packages loaded successfully.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# Field Categorization mapping
fields = {
    "Features": ["imp_prev15", "clk_prev15", "pos_prev15", "ga4_sessions_prev15", "word_count"],
    "Label": ["is_declining (imp_future15 < 0.8 * imp_prev15)"],
    "Context": ["client_hash_id", "content_hash_id", "report_date"],
    "Excluded": ["trend_direction", "trend_pct"]
}
for k, v in fields.items():
    print(f"{k}: {', '.join(v)}")


Features: imp_prev15, clk_prev15, pos_prev15, ga4_sessions_prev15, word_count
Label: is_declining (imp_future15 < 0.8 * imp_prev15)
Context: client_hash_id, content_hash_id, report_date
Excluded: trend_direction, trend_pct


## 3. Verify it with queries (grain, counts, missing values, windows)

### **Verification Queries:**

- **Query 1 (Grain Check):** Proves that grouping by `report_date`, `client_hash_id`, and `content_hash_id` yields no duplicate rows.
- **Query 2 (Count and Date Span):** Measures the row count and date boundaries for our target mid-panel partition (`month=2026-03`).
- **Query 3 (Availability Check):** Filters with `IS TRUE` on `ga4_data_available` to show how many rows survive.

### **Five-Feature Frame:**
1. `imp_prev15`: *Knowable at decision moment* because it represents historical organic visibility prior to the split date.
2. `clk_prev15`: *Knowable at decision moment* because it represents historical click-through traffic prior to the split date.
3. `pos_prev15`: *Knowable at decision moment* because it represents historical average visibility position prior to the split date.
4. `ga4_sessions_prev15`: *Knowable at decision moment* because it represents user engagement sessions tracked by GA4 prior to the split date.
5. `word_count`: *Knowable at decision moment* because it represents the static text length of the content published on the site.

### **The Leakage Trap (Experiment):**
- **Leaked feature:** `imp_future15` (future impressions). Since our classification label `is_declining` is defined as `imp_future15 < 0.8 * imp_prev15`, including it makes the model's accuracy/F1-score jump toward 0.97 (or near perfect).
- **Honest feature set:** Deletes `imp_future15` and evaluates only the historical features, yielding a realistic and honest baseline accuracy.

In [3]:
# Run the 3 contract verification queries on March 2026 partition

# 1. Grain verification query
print("=== Query 1: Grain Verification ===")
grain_query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
"""
res_grain = con.execute(grain_query).fetchall()
print("Grain check result (should be empty if grain is correct):", res_grain)

# 2. Row count and date span query
print("\n=== Query 2: Row Count and Date Span ===")
count_query = f"""
SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')
"""
res_count = con.execute(count_query).fetchall()
print(f"Total Rows: {res_count[0][0]:,}")
print(f"Min Date: {res_count[0][1]}")
print(f"Max Date: {res_count[0][2]}")

# 3. Availability check query (filter with IS TRUE)
print("\n=== Query 3: GA4 Data Availability ===")
avail_query = f"""
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
       AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) as ga4_available_pct
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')
"""
res_avail = con.execute(avail_query).fetchall()
print(f"Total Rows: {res_avail[0][0]:,}")
print(f"GA4 Available Rows (IS TRUE): {res_avail[0][1]:,}")
print(f"GA4 Available Percentage: {res_avail[0][2] * 100:.4f}%")


=== Query 1: Grain Verification ===
Grain check result (should be empty if grain is correct): []

=== Query 2: Row Count and Date Span ===
Total Rows: 9,841,378
Min Date: 2026-03-01
Max Date: 2026-03-31

=== Query 3: GA4 Data Availability ===
Total Rows: 9,841,378
GA4 Available Rows (IS TRUE): 413,966
GA4 Available Percentage: 4.2064%


In [4]:
# 5-Feature Frame construction and Leakage Experiment
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("=== Building 5-Feature Frame & Leakage Query ===")
query = f"""
WITH feature_window AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) as imp_prev15,
           SUM(gsc_clicks) as clk_prev15,
           AVG(gsc_avg_position) as pos_prev15,
           SUM(ga4_sessions) as ga4_sessions_prev15
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')
    WHERE report_date <= '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),
label_window AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) as imp_future15
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')
    WHERE report_date > '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
)
SELECT fw.client_hash_id, fw.content_hash_id,
       fw.imp_prev15, fw.clk_prev15, fw.pos_prev15, fw.ga4_sessions_prev15,
       COALESCE(c.word_count, 0) as word_count,
       COALESCE(lw.imp_future15, 0) as imp_future15
FROM feature_window fw
LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
  ON fw.content_hash_id = c.content_hash_id
LEFT JOIN label_window lw
  ON fw.client_hash_id = lw.client_hash_id AND fw.content_hash_id = lw.content_hash_id
WHERE fw.imp_prev15 >= 50
"""

df = con.execute(query).df()
df = df.dropna(subset=['imp_prev15', 'clk_prev15', 'pos_prev15', 'ga4_sessions_prev15', 'word_count'])
print(f"Loaded feature frame with {len(df):,} rows.")
print("Columns in feature frame:", list(df.columns))

# Define label
df['is_declining'] = (df['imp_future15'] < 0.8 * df['imp_prev15']).astype(int)

# Honest features vs Leaked features
honest_cols = ['imp_prev15', 'clk_prev15', 'pos_prev15', 'ga4_sessions_prev15', 'word_count']
leaked_cols = honest_cols + ['imp_future15'] # imp_future15 is directly used to calculate is_declining!

X_honest = df[honest_cols]
X_leaked = df[leaked_cols]
y = df['is_declining']

X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)

print("\n--- Leaked Model Performance (with imp_future15) ---")
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leak.fit(X_tr_l, y_tr)
preds_leak = model_leak.predict(X_te_l)
print(classification_report(y_te, preds_leak, digits=4))

print("\n--- Honest Model Performance (without imp_future15) ---")
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_honest.fit(X_tr_h, y_tr)
preds_honest = model_honest.predict(X_te_h)
print(classification_report(y_te, preds_honest, digits=4))

print("Baseline rate (always predict majority):", max(y.mean(), 1 - y.mean()))


=== Building 5-Feature Frame & Leakage Query ===
Loaded feature frame with 49,273 rows.
Columns in feature frame: ['client_hash_id', 'content_hash_id', 'imp_prev15', 'clk_prev15', 'pos_prev15', 'ga4_sessions_prev15', 'word_count', 'imp_future15']

--- Leaked Model Performance (with imp_future15) ---
              precision    recall  f1-score   support

           0     0.9677    0.9918    0.9796      9025
           1     0.9759    0.9092    0.9414      3294

    accuracy                         0.9697     12319
   macro avg     0.9718    0.9505    0.9605     12319
weighted avg     0.9699    0.9697    0.9694     12319


--- Honest Model Performance (without imp_future15) ---
              precision    recall  f1-score   support

           0     0.7606    0.9260    0.8352      9025
           1     0.4981    0.2013    0.2867      3294

    accuracy                         0.7322     12319
   macro avg     0.6293    0.5636    0.5609     12319
weighted avg     0.6904    0.7322    0.6885

## 4. Data limits

### **Named Limitation of this Slice:**

**GA4 Data Tracking Sparsity (Unbalanced GA4 Integration):**
As proven by our third query, only **4.21%** of the daily performance rows for March 2026 have `ga4_data_available = TRUE`. The remaining 95.79% of rows are zero-filled. Under the hood, this means that for the vast majority of clients or dates in this partition, GA4 integration was not active or set up yet. If we build features like `ga4_sessions_prev15` and include them in a global model, the model will treat these missing integrations as "zero user interest" instead of "not measured," introducing severe bias. Thus, any engagement model is restricted to a small, filtered subset of clients with active GA4 tracking.

In [5]:
# Verify GA4 missingness/zeros
zeros_ga4_pct = (df['ga4_sessions_prev15'] == 0).mean() * 100
print(f"Percentage of rows in our filtered slice where ga4_sessions_prev15 is 0: {zeros_ga4_pct:.2f}%")


Percentage of rows in our filtered slice where ga4_sessions_prev15 is 0: 33.33%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.